# #236 Historical European 1X2 Market Proxy

This executed notebook profiles the Nowscore/Titan-backed `beidan_xlsm` lane. `initial`, `10h`, and `1h` are **RETROSPECTIVE_RELATIVE_PROXY** labels; no output is Beijing SP or strict-forward evidence.

In [1]:
import json
import os
from pathlib import Path

DATA_ROOT = Path(os.environ.get('BEIJING_LOT_DATA_ROOT', '/home/drwho1985/workspace/soccer/beijing-lot/data'))
PACKAGE = DATA_ROOT / 'derived' / 'bj236' / 'eu_odds_proxy_v1'
manifest = json.loads((PACKAGE / 'MANIFEST.json').read_text(encoding='utf-8'))
facts = json.loads((PACKAGE / 'report_facts.json').read_text(encoding='utf-8'))
assert manifest['contract'] == 'BJ-UNIFIED-EU-ODDS-TIMESERIES-001'
assert manifest['market_type'] == 'euro_1x2'
assert manifest['causal_grade'] == 'RETROSPECTIVE_RELATIVE_PROXY'
facts

{'causal_grade': 'RETROSPECTIVE_RELATIVE_PROXY',
 'exclusion_rows': 986,
 'market_type': 'euro_1x2',
 'matched_fixture_count': 192443,
 'probability_rows': 1731423,
 'proxy_rows': 1731423,
 'source_sha256': '7f62079b18b3c028884da92aa8393f5b198246c9d7c32cd0b401562a055c5d6f',
 'strict_forward_rows': 0,
 'trace_rows': 578127,
 'translation_rows': 577141}

In [2]:
import pandas as pd

probability_parts = sorted((PACKAGE / 'probabilities').glob('*.parquet'))
translation_parts = sorted((PACKAGE / 'translation_readiness').glob('*.parquet'))
probabilities = pd.concat([pd.read_parquet(p) for p in probability_parts], ignore_index=True)
translation = pd.concat([pd.read_parquet(p) for p in translation_parts], ignore_index=True)
assert set(probabilities['market_type']) == {'euro_1x2'}
assert set(probabilities['causal_grade']) == {'RETROSPECTIVE_RELATIVE_PROXY'}
assert (probabilities.groupby(['canonical_fixture_key', 'snapshot_kind'])['probability'].sum().sub(1).abs() < 1e-12).all()
coverage = probabilities.groupby('snapshot_kind')['canonical_fixture_key'].nunique().rename('matched_fixture_count').reset_index()
translation_counts = translation.groupby('translation_status').size().rename('count').reset_index()
coverage, translation_counts

(  snapshot_kind  matched_fixture_count
 0           10h                 192297
 1            1h                 192438
 2       initial                 192406,
                                   translation_status   count
 0     UNAVAILABLE_MISSING_OR_CONFLICTING_NATIVE_LINE    7746
 1  UNAVAILABLE_NONZERO_LINE_REQUIRES_GOAL_DIFFERENCE  569395)